# **Bag of Words (BOW) Text Classification**

This notebook demonstrates how to use TFIDF Vectorizer from scikit-learn

_______________

### **Set-Up**

This section focuses on loading the data, python libraries (which will be used throughout this notebook)

In [596]:
# Loading Libraries
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("--"*50)
print("Libraries loaded successfully.")
print("--"*50)

----------------------------------------------------------------------------------------------------
Libraries loaded successfully.
----------------------------------------------------------------------------------------------------


In [597]:
# Loading the dataset
data = pd.read_csv("data/english_pages_metadata_clean_with_labels.csv")

print("--"*50)
print("First few Rows of the Dataset:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
First few Rows of the Dataset:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,_merge
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,both
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,both
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,both
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,both
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...",both
...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,both
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,both
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,both
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,both


In [598]:
data.drop(columns=["_merge"], inplace=True)

In [599]:
# Dataset Shape:
print("--"*50)
print("Dataset Shape (Rows, Columns)")
print("--"*50)
data.shape

----------------------------------------------------------------------------------------------------
Dataset Shape (Rows, Columns)
----------------------------------------------------------------------------------------------------


(596, 6)

In [600]:
# Dataset Information
print("--"*50)
print("Dataset Information")
print("--"*50)
data.info()

----------------------------------------------------------------------------------------------------
Dataset Information
----------------------------------------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 596 entries, 0 to 595
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   page_id             596 non-null    float64
 1   assigned_to         596 non-null    object 
 2   manual_label        596 non-null    object 
 3   manual_label_clean  596 non-null    object 
 4   manual_label_final  596 non-null    object 
 5   full_text           596 non-null    object 
dtypes: float64(1), object(5)
memory usage: 28.1+ KB


In [601]:
# Identifying NULLs and NaNs in the data set
print("--"*50)
print("Null Values in the dataset")
print("--"*50)
print(data.isnull().sum())

----------------------------------------------------------------------------------------------------
Null Values in the dataset
----------------------------------------------------------------------------------------------------
page_id               0
assigned_to           0
manual_label          0
manual_label_clean    0
manual_label_final    0
full_text             0
dtype: int64


In [602]:
print("--"*50)
print("Data Description for Numeric features:")
print("--"*50)
data.describe(include=[np.number])

----------------------------------------------------------------------------------------------------
Data Description for Numeric features:
----------------------------------------------------------------------------------------------------


,page_id
count,596.000000
mean,675.904362
std,449.207144
min,1.000000
25%,356.250000
50%,620.500000
75%,977.250000
max,2060.000000


Based on the above information, it is clear from a bird’s-eye view that there are no null or NaN (except for manual_label) values or even duplication of rows within the dataset.

____________

### **1. Text Preprocessing**

In this section, the primary focus is on cleaning the text data.Identifying and removing these characters early is important, as they can cause issues later in the pipeline and can negatively impact stability and compatibility. The primary goal here is to retain ASCII characters, such as English letters, numbers, and common punctuation (e.g., '(', ')', '[', ']', '+', '-', ' '), while ensuring that the existing structure of the text remains unchanged.

#### **1.1 Filtering out the English Words**

Here we will focusing on removing all the non-english character and preserve currance symbols.  `\x00-\x7f` - ASCII Range: It starts from ASCII 00 and ends at ASCII 127. Here the "\x" is an escape sequence telling the regex engine that the next two character are a hexadecimal, "00" means the start the first ASCII value and "7f" is the hexadecimal value for 127.

In [603]:
# list of currency symbols (few of them might not be present, but still we will keep it)
currency_symbols = [
    # Paired (Multi-character) symbols (Longest first)
    'USD','AU$', 'C$', 'NZ$', 'HK$', 'S$', 'US$', 'NT$', 'MX$', 'R$', 'zł', 'Kč','₨',
    'kr', 'Ft', 'ден', 'р.', 'лв', '₡', '₢', '₣', '₥', 

    # Single character symbols (Specific before generic)
    '€', '£', '¥', '元', '₹', '฿', '₩', '₽', '₪', '₺', '₦', '₵', '₫', 
    '₱', '₭', '₮', '₼', '₸', '₴', '៛', '₲', '₡', '₾', '֏', '﷼', 'Ξ', 'Ł', '$'
]

curr_symbols = ''.join(re.escape(s) for s in currency_symbols)
# function to clean the full text column
def full_text_clean (text):
    #Regex Pattern to indentify all the ASCII characters while retaining currency symbols
    re_pattern = rf"[\x00-\x7F{curr_symbols}]+"

    clean = re.findall(re_pattern, text)

    return clean

In [604]:
data['full_text_clean'] = data['full_text'].apply(full_text_clean)

print("--"*50)
print("New datastructure after cleaning the full_text columns:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
New datastructure after cleaning the full_text columns:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , A..."
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea..."
...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013\n3patchcrafts\nTh...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...


#### **1.2 Removal of Emails**

Removing email addresses because they are primarily alphanumeric in nature and do not carry meaningful semantic information. They also do not contribute as reliable features for text classification and may instead introduce noise into the model

In [605]:
def extract_emails(text):
    """
    Removing all email addresses from text
    """
    # Regex pattern for emails
    email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
    clean_text = []
    
    # Find all matches
    for i in range (len(text)):
        clean_text.append(re.sub(email_pattern, '', text[i]))
    
    return clean_text

In [606]:
data['full_text_clean'] = data['full_text_clean'].apply(extract_emails)

print("--"*50)
print("Email Removal:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
Email Removal:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , A..."
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea..."
...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013\n3patchcrafts\nTh...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...


#### **1.3 Removel of Phone Numbers**

Phone numbers do not carry meaningful semantic information for text classification, so it is better to remove them rather than retain them.

In [607]:
def extract_phones(text):
    """
    Extract phone numbers in various formats
    Returns: list of phone numbers
    """
    clean_text = []
    
    # Pattern 1: 555-123-4567
    pattern1 = r'\b\d{3}-\d{3}-\d{4}\b'
    # Pattern 2: (555) 123-4567
    pattern2 = r'\(\d{3}\)\s*\d{3}-\d{4}'
    # Pattern 3: 800-555-0123 (toll-free)
    pattern3 = r'\b[8-9]00-\d{3}-\d{4}\b'
    # Pattern 4: +1-555-123-4567
    pattern4 = r'\+{1,3}-\d{3}-\d{3}-\d{4}'
    # Pattern 5 : +267 7610 0890
    pattern5 = r'\+\d{3}\s*\d{4}\s*\d{4}'

    for i in range (len(text)):
        clean_text.append(re.sub(pattern1, '', text[i]))
        clean_text.append(re.sub(pattern2, '', text[i]))
        clean_text.append(re.sub(pattern3, '', text[i]))
        clean_text.append(re.sub(pattern4, '', text[i]))
        clean_text.append(re.sub(pattern5, '', text[i]))
           
    return clean_text

In [608]:
data['full_text_clean'] = data['full_text_clean'].apply(extract_phones)

print("--"*50)
print("Phone Number Removal:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
Phone Number Removal:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , Fa..."
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea..."
...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013\n3patchcrafts\nTh...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...


#### **1.4 Removing Dates**

Dates may add some contextual information, they do not carry meaningful semantic value for text classification and therefore do not meaningfully contribute to the model.

In [609]:
def extract_dates(text):
    """
    Extract phone numbers in various formats
    Returns: list of phone numbers
    """
    clean_text = []
    
    # Pattern 1: YYYY-MM-DD
    pattern1 = r'\d{4}-\d{2}-\d{2}'
    # Pattern 2: DD-MM-YYYY or MM-DD-YYYY
    pattern2 = r'\d{2}-\d{2}-\d{4}'
    # Pattern 3: MM/DD/YYYY or DD/MM/YYYY
    pattern3 = r'\d{2}/\d{2}/\d{4}'
    # Pattern 4: January 15, 2024 or Jan 15, 2024
    pattern4 = r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\s\d{1,2},\s\d{4}'
    # Pattern 5: 18 May. 2025 | 17 May 2025
    pattern5 = r'\d{1,2}\s(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\.?\s\d{4}'
    # pattern 6: 2024.01.15
    pattern6 = r'\d{4}\.\d{2}\.\d{2}'

    for i in range (len(text)):
        clean_text.append(re.sub(pattern1, '', text[i]))
        clean_text.append(re.sub(pattern2, '', text[i]))
        clean_text.append(re.sub(pattern3, '', text[i]))
        clean_text.append(re.sub(pattern4, '', text[i]))
        clean_text.append(re.sub(pattern5, '', text[i]))
        clean_text.append(re.sub(pattern6, '', text[i]))
           
    return clean_text

In [610]:
data['full_text_clean'] = data['full_text_clean'].apply(extract_dates)

print("--"*50)
print("Date Removal:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
Date Removal:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , Fa..."
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea..."
...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013\n3patchcrafts\nTh...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...


#### **1.5 Removing all the URLs**

Removing URLs from the text will reduce its overall length. In turn, this helps us retain only the most relevant content for text classification.

In [611]:
def extract_urls(text):
    """
    Remove URLs from text while preserving everything else
    """
    url_pattern = r'https?://\S+|www\.\S+'

    clean_text = []
    # Find all matches
    for i in range (len(text)):
        clean_text.append(re.sub(url_pattern, '', text[i]))
    
    return clean_text

In [612]:
data['full_text_clean'] = data['full_text_clean'].apply(extract_urls)

print("--"*50)
print("URL Removal:")
print("--"*50)
data 

----------------------------------------------------------------------------------------------------
URL Removal:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , Fa..."
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea..."
...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013\n3patchcrafts\nTh...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...


#### **1.6 Replacing `\n` with Space**

Replacing `\n` with a space removes the ambiguity caused when text appears on a new line but is recorded as `\n`, resulting in a single continuous text line.

In [613]:
# Cleaning "\n" from the cleaned full_text column
def clean_text(text):
    clean = r"[\r\n]+"

    sentence = []

    # Find all matches
    for i in range (len(text)):
        sentence.append(re.sub(clean, " ", text[i]))

    sentence = [stn for stn in sentence if stn]

    return sentence

data['full_text_clean'] = data['full_text_clean'].apply(clean_text)

print("--"*50)
print("New datastructure after cleaning the full_text columns:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
New datastructure after cleaning the full_text columns:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , Fa..."
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea..."
...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013 3patchcrafts The ...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...


In [614]:
# Random index number to check how a full_text field and full_text_clean field looks like
idx = np.random.randint(0, 596)
print("--"*50)
print(f"Text Field ({idx}) -- Total Character Length ({len(data.loc[idx, 'full_text_clean'])})")
print("--"*50)
print(data.loc[idx, 'full_text_clean'])

----------------------------------------------------------------------------------------------------
Text Field (591) -- Total Character Length (330)
----------------------------------------------------------------------------------------------------
['Blog post Archives - 38th VoyageMystic Seaport The 38th Voyage of the CHARLES W. MORGAN Search Navigation Search Voyage Home The 38th Voyage: Introduction & Sitemap Further Reading Glossary Ship Stories People Places More! Visit the MORGAN Today G. W. Blunt White Library Mystic Seaport for Educators The Charles W. Morgan Book Web Store Timeline Blog post Show All Shuffle Any StoriesHistory and AncestrySailing the ShipScience and Conservation Any PeopleCaptain and crewHostsMuseum staffMuseum staff: Alexandra McInturfMuseum staff: Tom DanielsShipwrightsStowawayVoyagersVoyagers: Karim Tiro Any PlacesBostonFrom Boston through the Cape Cod Canal: July 23Mass. Maritime Academy to New London: July 29-30Massachusetts Maritime AcademyMystic Seapo

As you can see, we currently have a list-of-lists structure, which can be difficult to manage in the subsequent steps of the execution. Therefore, we will combine it into a single list for simplicity and easier processing.

In [615]:
data['full_text_merge'] = data['full_text_clean'].apply(lambda x: ' '.join(x))

print("--"*50)
print("New datastructure after combining the list-of-list structure into one single list:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
New datastructure after combining the list-of-list structure into one single list:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean,full_text_merge
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , Fa...",Fall arrest and work positioning harness Fall...
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...,Vermont Mountain Eats: Jay Peak - All Mountain...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...,05/18/2020 Booking Report for Bulloch County -...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...,Is Pickleball Easier than Tennis? | AllRacket ...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea...","Account Executive, Auto Finance - Greater Seat..."
...,...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...,Blog post Archives - 38th VoyageMystic Seaport...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...,Kids bike pedals collection 3D Model in Bicycl...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013 3patchcrafts The ...,3patchcrafts: February 2013 3patchcrafts The p...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...,Fatal error: Uncaught mysqli_sql_exception: Ta...


In [616]:
idx = np.random.randint(0, 596)
print("--"*50)
print(f"Text Field ({idx}) -- Total Character Length ({len(data.loc[idx, 'full_text_merge'])})")
print("--"*50)
print(data.loc[idx, 'full_text_merge'])

----------------------------------------------------------------------------------------------------
Text Field (127) -- Total Character Length (62969)
----------------------------------------------------------------------------------------------------


In [617]:
print("--"*50)
print(f"Text Field ({idx}) -- Total Character Length ({len(data.loc[idx, 'full_text_clean'])})")
print("--"*50)
print(data.loc[idx, 'full_text_clean'])

----------------------------------------------------------------------------------------------------
Text Field (127) -- Total Character Length (30)
----------------------------------------------------------------------------------------------------
["Warning: require(/home/c9921708/public_html/angelstouch444.com/wp-includes/functions.php): Failed to open stream: No such file or directory in /home/c9921708/public_html/angelstouch444.com/wp-settings.php on line 115 Fatal error: Uncaught Error: Failed opening required '/home/c9921708/public_html/angelstouch444.com/wp-includes/functions.php' (include_path='.:/opt/alt/php84/usr/share/pear:/opt/alt/php84/usr/share/php:/usr/share/pear:/usr/share/php') in /home/c9921708/public_html/angelstouch444.com/wp-settings.php:115 Stack trace: #0 /home/c9921708/public_html/angelstouch444.com/wp-config.php(110): require_once() #1 /home/c9921708/public_html/angelstouch444.com/wp-load.php(50): require_once('/home/c9921708/...') #2 /home/c9921708/public_htm

#### **1.7 Normalizing Text**

Here we will convert all the sentences in the lower case and remove extra spaces and punctuation using regex expression

In [618]:
def lowercase_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    return text.lower()

In [619]:
data['full_text_merge'] = data['full_text_merge'].apply(lowercase_text)

print("--"*50)
print("New datastructure after converting the full_text_merge column to lowercase:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
New datastructure after converting the full_text_merge column to lowercase:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean,full_text_merge
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , Fa...",fall arrest and work positioning harness fall ...
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...,vermont mountain eats jay peak all mountain ma...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...,05 18 2020 booking report for bulloch county a...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...,is pickleball easier than tennis allracket ski...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea...",account executive auto finance greater seattle...
...,...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...,blog post archives 38th voyagemystic seaport t...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...,kids bike pedals collection 3d model in bicycl...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013 3patchcrafts The ...,3patchcrafts february 2013 3patchcrafts the pl...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...,fatal error uncaught mysqli_sql_exception tabl...


In [620]:
# Sanity check for the full_text_merge column
idx = np.random.randint(0, 596)
print("--"*50)
print(f"Text Field ({idx}) -- Total Character Length ({len(data.loc[idx, 'full_text_merge'])})")
print("--"*50)
print(data.loc[idx, 'full_text_merge'])

----------------------------------------------------------------------------------------------------
Text Field (581) -- Total Character Length (363869)
----------------------------------------------------------------------------------------------------
top ielts coaching in ahmedabad archives 1businessworld skip to navigation skip to content 1businessworld search for search 1businessworld about 1bw entrepreneurship privacy policy copyright intellectual property policy opt out preferences contact us business solutions growth hub professional profile digital business card global business profile growth pioneers growthceo 1navigator events global retail conference leading entrepreneurs of the world marketplace global business central legal hub menu 1businessworld about 1bw entrepreneurship privacy policy copyright intellectual property policy opt out preferences contact us business solutions growth hub professional profile digital business card global business profile growth pioneers gro

In [621]:
print("--"*50)
print(f"Text Field ({idx}) -- Total Character Length ({len(data.loc[idx, 'full_text_clean'])})")
print("--"*50)
print(data.loc[idx, 'full_text_clean'])

----------------------------------------------------------------------------------------------------
Text Field (581) -- Total Character Length (90)
----------------------------------------------------------------------------------------------------
['Top IELTS Coaching in Ahmedabad Archives | 1BusinessWorld Skip to navigation Skip to content 1BusinessWorld Search for: Search 1BusinessWorld About 1BW Entrepreneurship Privacy Policy Copyright & Intellectual Property Policy Opt-out preferences Contact Us Business Solutions Growth Hub Professional Profile | Digital Business Card Global Business Profile Growth Pioneers GrowthCEO 1Navigator Events Global Retail Conference Leading Entrepreneurs of the World Marketplace Global Business Central Legal Hub Menu 1BusinessWorld About 1BW Entrepreneurship Privacy Policy Copyright & Intellectual Property Policy Opt-out preferences Contact Us Business Solutions Growth Hub Professional Profile | Digital Business Card Global Business Profile Growth Pio

#### **1.8 Removal of Stop Words**

These are common repeating words in a sentence, such as “a,” “the,” “he,” and “there.” These words typically carry little semantic meaning on their own, as they primarily serve grammatical functions. In this assignment, where our goal is to identify meaningful classification terms, such stopwords would dominate the corpus because they appear in nearly all sentences, thereby adding noise rather than useful information.

In [622]:
# loading stopwords from both sklearn and nltk
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.corpus import stopwords

nltk_stopwords = set(stopwords.words('english'))
sklearn_stopwords = set(ENGLISH_STOP_WORDS)

print("--"*50)
print("Number of NLTK stopwords:",len(nltk_stopwords), "\nStopwords from NLTK:", list(nltk_stopwords)[:10], "...")
print("--"*50)
print("Number of Sklearn stopwords:",len(sklearn_stopwords), "\nStopwords from Sklearn:", list(sklearn_stopwords)[:10], "...")
print("--"*50)

----------------------------------------------------------------------------------------------------
Number of NLTK stopwords: 198 
Stopwords from NLTK: ["you'd", "hadn't", "they'd", 'yours', 'than', 'does', 'he', 'this', 'on', 's'] ...
----------------------------------------------------------------------------------------------------
Number of Sklearn stopwords: 318 
Stopwords from Sklearn: ['yours', 'often', 'than', 'show', 'he', 'latter', 'un', 'get', 'neither', 'her'] ...
----------------------------------------------------------------------------------------------------


In [623]:
# Chceking the common stopwords between both libraries
common_stopwords = nltk_stopwords.intersection(sklearn_stopwords)
print("--"*50)
print("Number of Common Stopwords:", len(common_stopwords), "\nCommon Stopwords:", list(common_stopwords)[:10], "...")
print("--"*50)

----------------------------------------------------------------------------------------------------
Number of Common Stopwords: 119 
Common Stopwords: ['yours', 'than', 'he', 'this', 'on', 'is', 'her', 'can', 'nor', 'from'] ...
----------------------------------------------------------------------------------------------------


**Take Away:**

Since the default stopword list in NLTK is predefined and not intended to be modified directly in place, it offers limited flexibility for customization. In contrast, scikit-learn’s stopword list can be easily copied and extended (for example, by adding or removing specific words such as negations) to better suit a particular corpus.

Therefore, we will use the scikit-learn stopword list as the base and extend it by incorporating selected stopwords from NLTK, along with additional domain-specific terms that may negatively affect the classification context based on our domain understanding.

In [624]:
# Chceking the common stopwords between both libraries
nltk_unique = nltk_stopwords.difference(sklearn_stopwords)
print("--"*50)
print("Number of NLTK Unique Stopwords:", len(nltk_unique), "\nNLTK Unique Stopwords:", list(nltk_unique)[:10], "...")
print("--"*50)

----------------------------------------------------------------------------------------------------
Number of NLTK Unique Stopwords: 79 
NLTK Unique Stopwords: ["you'd", "hadn't", "they'd", 'does', 's', 'o', "it'd", 'did', 'd', 've'] ...
----------------------------------------------------------------------------------------------------


In [ ]:
# List of common words to be removed from full_text_clean column
common_word_list = {'followers', 'instagram', 'facebook', 'like', 'messages', 'support',
                    'sitemap', 'free', 'faq', 'twitter', 'search', 'subscribe', 'youtube', 
                    'feedback', 'linkedin', 'demo', 'menu', 'following', 'unsubscribe', 
                    'snapchat', 'tiktok', 'whatsapp', 
                    'profile', 'share', 'reddit', 'download', 'settings', 'notifications', 'qa', 'people'}

# Common phrases to remove
phrase_list = ["find out more", "join now","terms & conditions", "terms and conditions","all rights reserved","free trial", 
             "click here", "about us", "contact us","terms of service", "privacy policy", "help center", "our story","our team",
             "read more", "learn more", "get started", "download now", "sign up", "log in"]

In [626]:
# Combining sklearn and nltk stop words and adding the common words to the stop word list
stop_words_corpus = sklearn_stopwords.union(nltk_unique).union(common_word_list).union(phrase_list)

print("--"*50)
print("Total Stop Words in the Corpus:", len(stop_words_corpus), "\nSample Stop Words:", list(stop_words_corpus)[:10], "...")
print("--"*50)

----------------------------------------------------------------------------------------------------
Total Stop Words in the Corpus: 448 
Sample Stop Words: ['yours', "hadn't", 'often', 'than', 'show', 'menu', 'does', 'he', 's', 'latter'] ...
----------------------------------------------------------------------------------------------------


In [627]:
# function to remove stop words from the full_text_merge column
def remove_stop_words(text):
    clean_text = []
    for word in text.split():
        if word not in stop_words_corpus:
            clean_text.append(word)
    
    return ' '.join(clean_text)

In [628]:
data['analysis_text'] = data['full_text_merge'].apply(remove_stop_words)

print("--"*50)
print("New datastructure after removing stop words from the full_text_merge column:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
New datastructure after removing stop words from the full_text_merge column:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean,full_text_merge,analysis_text
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , Fa...",fall arrest and work positioning harness fall ...,fall arrest work positioning harness fall arre...
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...,vermont mountain eats jay peak all mountain ma...,vermont mountain eats jay peak mountain mamas ...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...,05 18 2020 booking report for bulloch county a...,05 18 2020 booking report bulloch county allon...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...,is pickleball easier than tennis allracket ski...,pickleball easier tennis allracket skip conten...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea...",account executive auto finance greater seattle...,account executive auto finance greater seattle...
...,...,...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...,blog post archives 38th voyagemystic seaport t...,blog post archives 38th voyagemystic seaport 3...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...,kids bike pedals collection 3d model in bicycl...,kids bike pedals collection 3d model bicycle 5...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013 3patchcrafts The ...,3patchcrafts february 2013 3patchcrafts the pl...,3patchcrafts february 2013 3patchcrafts place ...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...,fatal error uncaught mysqli_sql_exception tabl...,fatal error uncaught mysqli_sql_exception tabl...


In [629]:
# Sanity check for the analysis_text column with full_text_merge column and full_text_clean column
idx = np.random.randint(0, 596)
print("--"*50)
print(f"Text Field ({idx})")
print("--"*50)
print(f"Total Character Length for Analysis Ready Text Column ({len(data.loc[idx, 'analysis_text'])})")
print("--"*50)
print(f"Total Character Length for full_text_merge Column ({len(data.loc[idx, 'full_text_merge'])})")
print("--"*50)
print(f"Total Character Length for full_text_clean Column ({len(data.loc[idx, 'full_text_clean'])})-- because this column is a list of lists, the character length is much higher than the other two columns")
print("--"*50)


----------------------------------------------------------------------------------------------------
Text Field (175)
----------------------------------------------------------------------------------------------------
Total Character Length for Analysis Ready Text Column (278444)
----------------------------------------------------------------------------------------------------
Total Character Length for full_text_merge Column (355034)
----------------------------------------------------------------------------------------------------
Total Character Length for full_text_clean Column (210)-- because this column is a list of lists, the character length is much higher than the other two columns
----------------------------------------------------------------------------------------------------


### **2. TF-IDF Construction**

TF-IDF stands for Term Frequency-Inverse Document Frequency. Here, we will focus on converting our preprocessed text corpus `(analysis_text)` into a numerical feature matrix, where each document is represented as a high-dimensional vector of TF-IDF scores.

In [630]:
# Defininfg TFIDF Vectorizer
vectorizer = TfidfVectorizer(
    max_features=500,        # total features, NOT per document
    ngram_range=(1, 2),      # unigrams + bigrams
    max_df=0.95,             # ignore very common words
    min_df=2                 # ignore rare noise
)